<a href="https://colab.research.google.com/github/bquast/colab/blob/master/qwen3_1_7b-SFT-STE-v0.0.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 41.1 MB/s eta 0:00:00


In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

2.11.0+cu128
True
NVIDIA A100-SXM4-40GB
39.5


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen3-1.7B-Base"

tokenizer = AutoTokenizer.from_pretrained(name)

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    name,
    dtype=dtype
).cuda()

print(sum(p.numel() for p in model.parameters()))
print(dtype)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

1720574976
torch.bfloat16


In [4]:
prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation is a general increase in prices and fall in the purchasing value of money. It is a key economic indicator that reflects the overall health of an economy. Inflation can be caused by various factors, including:
1. Demand-pull inflation: This occurs when the demand for goods and services exceeds


In [5]:
from datasets import load_dataset, concatenate_datasets

old = load_dataset("bquast/ste-sft-v0.0.1")

new = load_dataset(
    "json",
    data_files="ste100_sft_source_dataset_v5_translated.jsonl",
    split="train"
)

bad_ids = {
    "003",  # major repair -> important repair
    "005",  # up to date -> correct
    "063",  # competent -> approved
    "069",  # appropriately rated -> approved
    "091",  # competent -> approved
    "094",  # up to date -> correct
    "103",  # source provenance failure
}

new = new.filter(
    lambda x: x["year"] <= 2022 and x["id"] not in bad_ids
)

print("new clean:", len(new))

train.jsonl:   0%|          | 0.00/31.2k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/3.71k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/42 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/84 [00:00<?, ? examples/s]

new clean: 49


In [6]:
new_split = new.train_test_split(
    test_size=5,
    seed=42
)

old_train = old["train"].select_columns(
    ["source_sentence", "ste_translation"]
)

old_test = old["test"].select_columns(
    ["source_sentence", "ste_translation"]
)

new_train = new_split["train"].select_columns(
    ["source_sentence", "ste_translation"]
)

new_test = new_split["test"].select_columns(
    ["source_sentence", "ste_translation"]
)

train_raw = concatenate_datasets([old_train, new_train])
test_raw = concatenate_datasets([old_test, new_test])

print("train:", len(train_raw))
print("test:", len(test_raw))

train: 86
test: 10


In [7]:
def format_example(x):
    return {
        "prompt": (
            "Rewrite in Simplified Technical English:\n"
            + x["source_sentence"]
            + "\n"
        ),
        "completion": x["ste_translation"] + tokenizer.eos_token
    }

train_data = train_raw.map(format_example)
test_data = test_raw.map(format_example)

train_data = train_data.select_columns(["prompt", "completion"])
test_data = test_data.select_columns(["prompt", "completion"])

print(train_data[0])

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

{'prompt': 'Rewrite in Simplified Technical English:\nFloor or ground surfaces shall be stable, firm, and slip resistant and shall comply with 302.\n', 'completion': 'Floor or ground surfaces must be stable, firm, and slip-resistant. They must comply with Section 302.<|endoftext|>'}


In [8]:
model.eval()

for x in test_data:
    inputs = tokenizer(
        x["prompt"],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    print("\nSOURCE:", x["prompt"].split("\n", 1)[1].strip())
    print("BASE:  ", tokenizer.decode(generated, skip_special_tokens=True))
    print("TARGET:", x["completion"].replace(tokenizer.eos_token, ""))


SOURCE: The utilization of a ground fault circuit interrupter is mandated for all electrical receptacles located within six feet of the outer edge of a sink.
BASE:   The use of a ground fault circuit interrupter is required for all electrical receptacles situated within six feet of the outer edge of a sink.
TARGET: You must use a ground fault circuit interrupter for all electrical receptacles that are not more than six feet from the outer edge of a sink.

SOURCE: The purpose of the heat exchanger is to facilitate the transfer of thermal energy from a high-temperature fluid to a lower-temperature fluid without allowing them to mix.
BASE:   The primary function of the heat exchanger is to transfer thermal energy from a high-temperature fluid to a lower-temperature fluid, ensuring no mixing occurs between the two.
TARGET: A heat exchanger transfers heat from a high-temperature fluid to a lower-temperature fluid so that the two fluids do not mix.

SOURCE: Calibration of the pressure trans

In [9]:
del model
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    name,
    dtype=torch.float32
).cuda()

print(next(model.parameters()).dtype)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

torch.float32


In [10]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qwen3-1.7b-ste-v0.0.2",
    num_train_epochs=8,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_length=512,
    bf16=use_bf16,
    fp16=not use_bf16,
    completion_only_loss=True,
    logging_steps=5,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_data,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/86 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.996523
10,0.614504
15,0.580601
20,0.457842
25,0.272689
30,0.122751
35,0.165693
40,0.166494
45,0.095055
50,0.032949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=176, training_loss=0.10504618555580175, metrics={'train_runtime': 181.9569, 'train_samples_per_second': 3.781, 'train_steps_per_second': 0.967, 'total_flos': 419254388158464.0, 'train_loss': 0.10504618555580175, 'entropy': 0.00012865703320130706, 'num_tokens': 43208.0, 'mean_token_accuracy': 1.0, 'epoch': 8.0})

In [11]:
model.eval()

for x in test_data:
    inputs = tokenizer(
        x["prompt"],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    print("\nSOURCE:", x["prompt"].split("\n", 1)[1].strip())
    print("SFT:   ", tokenizer.decode(generated, skip_special_tokens=True))
    print("TARGET:", x["completion"].replace(tokenizer.eos_token, ""))


SOURCE: The utilization of a ground fault circuit interrupter is mandated for all electrical receptacles located within six feet of the outer edge of a sink.
SFT:    You must use a ground fault circuit interrupter for all electrical receptacles that are near the outer edge of a sink.
TARGET: You must use a ground fault circuit interrupter for all electrical receptacles that are not more than six feet from the outer edge of a sink.

SOURCE: The purpose of the heat exchanger is to facilitate the transfer of thermal energy from a high-temperature fluid to a lower-temperature fluid without allowing them to mix.
SFT:    The purpose of the heat exchanger is to let thermal energy transfer from a high-temperature fluid to a lower-temperature fluid. It does not let them mix.
TARGET: A heat exchanger transfers heat from a high-temperature fluid to a lower-temperature fluid so that the two fluids do not mix.

SOURCE: Calibration of the pressure transducer should be carried out at regular interva

In [12]:
model.save_pretrained("qwen3-1.7b-ste-v0.0.2")
tokenizer.save_pretrained("qwen3-1.7b-ste-v0.0.2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-1.7b-ste-v0.0.2/tokenizer_config.json',
 'qwen3-1.7b-ste-v0.0.2/chat_template.jinja',
 'qwen3-1.7b-ste-v0.0.2/tokenizer.json')